# Baseline Evaluation — intent-classifier

This notebook:
1. Clones the `intent-classifier` repo from GitHub
2. Installs required Python packages
3. Authenticates with Hugging Face (needed for gated models like Gemma, Llama)
4. Configures paths and settings
5. Runs a **sanity check** (3 examples per model) for zero-shot and few-shot
6. Runs **zero-shot** full baseline evaluation across all models
7. Runs **few-shot** full baseline evaluation across all models
8. Downloads outputs for local analysis

> **Runtime:** Set runtime type to **GPU → T4** (or better) for reasonable speed.
> Analysis plots are generated locally from the downloaded reports using `plot_baselines.py`.


## 1. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")


## 2. Install dependencies

In [ ]:
%pip install -q \
    torch \
    transformers \
    accelerate \
    python-dotenv \
    sentencepiece \
    protobuf

print("Dependencies installed.")


## 3. Hugging Face authentication

Required for **gated models** (Gemma-3, Llama-3.2).  
Open-weight models (Qwen, SmolLM, TinyLlama) do **not** need a token.

Store your token in Colab Secrets as **`HF_TOKEN`** (left panel → 🔑 icon)  
or paste it in the cell below.

In [ ]:
import os

# ── Try Colab Secrets first (recommended) ────────────────────────────────────
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("HF_TOKEN secret not set — gated models (Gemma, Llama) will fail.")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        print("HF_TOKEN not found — gated models will fail.")
    else:
        print("HF_TOKEN already set in environment.")


## 4. Configuration

Paths and settings. Models are loaded automatically from `evaluation_lib` — no manual list needed.
Gated models (Gemma, Llama) require `HF_TOKEN` set above.


In [ ]:
import sys, os

REPO_DIR   = "/content/intent-classifier"
SRC_DIR    = f"{REPO_DIR}/evaluation_baseline/src"
DATA_FILE  = f"{REPO_DIR}/dataset_sample/sample.json"
ZS_OUT_DIR = f"{REPO_DIR}/evaluation_baseline/reports_zero_shot"
FS_OUT_DIR = f"{REPO_DIR}/evaluation_baseline/reports_few_shot"

# evaluation_lib lives at repo root
sys.path.insert(0, REPO_DIR)

from evaluation_lib.config import ALL_MODELS, MODEL_REGISTRY  # noqa: E402

DEVICE         = "auto"   # auto | cuda | cpu | mps
MAX_NEW_TOKENS = 32
SANITY_LIMIT   = 3        # examples per model for sanity check

os.makedirs(ZS_OUT_DIR, exist_ok=True)
os.makedirs(FS_OUT_DIR, exist_ok=True)

print(f"Models registered : {len(ALL_MODELS)}")
for k in ALL_MODELS:
    print(f"  {k:<20}  →  {MODEL_REGISTRY[k]}")
print(f"\nData file         : {DATA_FILE}")
print(f"Device            : {DEVICE}")
print(f"Max new tokens    : {MAX_NEW_TOKENS}")
print(f"Sanity check size : {SANITY_LIMIT} examples per model")


## 5. Sanity check — zero-shot (3 examples per model)

Runs each model on 3 examples in zero-shot mode.
Purpose: confirm every model loads and produces output before committing to the full run.
Reports are written to a separate `sanity_zero_shot/` directory so they don't pollute the full results.


In [ ]:
import subprocess, sys, os, json

SANITY_ZS_DIR = f"{REPO_DIR}/evaluation_baseline/sanity_zero_shot"
os.makedirs(SANITY_ZS_DIR, exist_ok=True)

EVAL_SCRIPT = f"{SRC_DIR}/baseline_eval.py"

def run_with_progress(model_key: str, mode: str, out_dir: str, limit: int | None = None) -> bool:
    """Run baseline_eval.py for one model and show dot-progress (one dot per 10%)."""
    cmd = [
        sys.executable, EVAL_SCRIPT,
        "--model",          model_key,
        "--data",           DATA_FILE,
        "--device",         DEVICE,
        "--mode",           mode,
        "--out-dir",        out_dir,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
    ]
    if limit is not None:
        cmd += ["--limit", str(limit)]

    model_id = MODEL_REGISTRY.get(model_key, model_key)
    print(f"\nRunning {model_id}")

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    assert proc.stdout is not None

    dots_shown = 0

    for raw_line in proc.stdout:
        line = raw_line.strip()
        # detect per-example progress line: "[  3/100] ..."
        if line.startswith("[") and "/" in line and "]" in line:
            try:
                fraction = line[1:line.index("]")]
                current, n_total = fraction.strip().split("/")
                pct      = int(current.strip()) / int(n_total.strip())
                dots_due = int(pct * 10)
                while dots_shown < dots_due:
                    print(".", end="", flush=True)
                    dots_shown += 1
            except (ValueError, IndexError):
                pass

    proc.wait()
    # fill remaining dots up to 10
    while dots_shown < 10:
        print(".", end="", flush=True)
        dots_shown += 1
    print("\ndone" if proc.returncode == 0 else f"\n[ERROR] exit code {proc.returncode}")
    return proc.returncode == 0


print("=== Sanity check — zero-shot ===\n")
sanity_zs_ok: list[str] = []
sanity_zs_fail: list[str] = []

for key in ALL_MODELS:
    ok = run_with_progress(key, mode="zero_shot", out_dir=SANITY_ZS_DIR, limit=SANITY_LIMIT)
    (sanity_zs_ok if ok else sanity_zs_fail).append(key)

print(f"\n✓ passed : {len(sanity_zs_ok)}")
if sanity_zs_fail:
    print(f"✗ failed : {sanity_zs_fail}")


## 6. Sanity check — few-shot (3 examples per model)

Same as above but in few-shot mode. Run this before the full few-shot evaluation.


In [ ]:
SANITY_FS_DIR = f"{REPO_DIR}/evaluation_baseline/sanity_few_shot"
os.makedirs(SANITY_FS_DIR, exist_ok=True)

print("=== Sanity check — few-shot ===\n")
sanity_fs_ok: list[str] = []
sanity_fs_fail: list[str] = []

for key in ALL_MODELS:
    ok = run_with_progress(key, mode="few_shot", out_dir=SANITY_FS_DIR, limit=SANITY_LIMIT)
    (sanity_fs_ok if ok else sanity_fs_fail).append(key)

print(f"\n✓ passed : {len(sanity_fs_ok)}")
if sanity_fs_fail:
    print(f"✗ failed : {sanity_fs_fail}")


## 7. Full evaluation — zero-shot

Runs all models on the full dataset in zero-shot mode.
Reports are saved to `evaluation_baseline/reports_zero_shot/`.
Previously completed models are automatically skipped (cached).


In [ ]:
from pathlib import Path

def latest_report_for(model_key: str, reports_dir: str) -> Path | None:
    matches = sorted(Path(reports_dir).glob(f"{model_key}_*.json"))
    return matches[-1] if matches else None


def run_full_eval(mode: str, out_dir: str) -> None:
    print(f"=== Full evaluation — {mode} ===\n")
    results: list[dict] = []

    for key in ALL_MODELS:
        cached = latest_report_for(key, out_dir)
        if cached:
            print(f"[CACHED] {key}  →  {cached.name}")
            results.append(json.loads(cached.read_text()))
            continue

        ok = run_with_progress(key, mode=mode, out_dir=out_dir, limit=None)
        if ok:
            report_path = latest_report_for(key, out_dir)
            if report_path:
                results.append(json.loads(report_path.read_text()))
        else:
            print(f"  [SKIP] {key} failed — continuing with next model")

    # ── Summary table ─────────────────────────────────────────────────────────
    if not results:
        print("No results to summarise.")
        return

    col = max(len(r["model_key"]) for r in results) + 2
    hdr = f"{'Model':<{col}}  {'Acc':>7}  {'Correct':>8}  {'Garbage%':>9}  {'AvgLat(ms)':>11}  {'Tok/s':>7}  {'Mem(MB)':>8}"
    sep = "-" * len(hdr)
    print(f"\n{'='*len(hdr)}\n  {mode.upper().replace('_','-')} SUMMARY\n{'='*len(hdr)}")
    print(hdr)
    print(sep)
    for r in sorted(results, key=lambda x: x["accuracy"], reverse=True):
        print(
            f"{r['model_key']:<{col}}  {r['accuracy']:>6.1%}  "
            f"{r['n_correct']:>4}/{r['n_examples']:<3}  "
            f"{r.get('garbage_pct', 0.0):>8.1f}%  "
            f"{r['avg_latency_ms']:>11.1f}  {r['avg_tokens_per_sec']:>7.1f}  "
            f"{r['peak_memory_mb']:>8.1f}"
        )
    print(sep)


run_full_eval("zero_shot", ZS_OUT_DIR)


## 8. Full evaluation — few-shot

Runs all models on the full dataset in few-shot mode.
Reports are saved to `evaluation_baseline/reports_few_shot/`.
Previously completed models are automatically skipped (cached).


In [ ]:
run_full_eval("few_shot", FS_OUT_DIR)


## 9. Download outputs

Zips all report directories and downloads them to your local machine.
Copy the downloaded archive into `evaluation_baseline/` and run `plot_baselines.py` locally to generate analysis plots.


In [ ]:
import shutil
from google.colab import files  # type: ignore

# Zip evaluation_baseline/reports_* directories
zip_base  = "/content/baseline_results"
zip_path  = zip_base + ".zip"

shutil.make_archive(
    zip_base,
    "zip",
    root_dir=f"{REPO_DIR}/evaluation_baseline",
    base_dir=".",
)
print(f"Archive created: {zip_path}")
print("Downloading…")
files.download(zip_path)
